# Modelo 3D de celdas – Etapa 1
Desarrollo y prueba de las ecuaciones. Mismo código que corre en GitHub Actions (`modelo3d/`).
**Resultado = estructura estimada, no medición.**

In [ ]:
import sys; sys.path.insert(0,'..')
import numpy as np, matplotlib.pyplot as plt
from modelo3d import termo, celda, fuentes

## 1. Sondeo del punto (Open-Meteo)
Cambiá lat/lon (ej. San Miguel de Tucumán).

In [ ]:
lat, lon = -26.83, -65.20
p,z,T,Td,u,v = fuentes.sondeo(lat, lon)
amb = termo.analizar(p,z,T,Td,u,v)
amb

## 2. Perfil: ambiente vs. parcela

In [ ]:
pc,pL = termo.parcela(p, p[0], T[0], Td[0])
plt.figure(figsize=(5,6)); plt.plot(T,z/1000,label='ambiente'); plt.plot(Td,z/1000,'--',label='rocío'); plt.plot(pc,z/1000,label='parcela')
plt.xlabel('°C'); plt.ylabel('km'); plt.legend(); plt.grid(alpha=.3)

## 3. Celdas reales del radar + satélite IR

In [ ]:
cs = fuentes.celdas('RMA11')
tbs, hora = fuentes.tb_minima([(c['lat'],c['lon']) for c in cs[:10]])
list(zip([c.get('max_dbz') for c in cs[:10]], tbs)), hora

## 4. Estructura 3D de una celda

In [ ]:
c = cs[0]; top = termo.altura_por_temperatura(z,T,tbs[0]) if tbs[0] is not None and tbs[0]<-30 else None
s = celda.modelar(c, amb, top)
zz=[a[0]/1000 for a in s['capas']]; dbz=[a[1] for a in s['capas']]; rr=[a[2] for a in s['capas']]
fig,ax=plt.subplots(1,2,figsize=(8,5)); ax[0].plot(dbz,zz); ax[0].set_xlabel('dBZ'); ax[0].set_ylabel('km'); ax[1].plot(rr,zz); ax[1].set_xlabel('radio km')
s['granizo'], s['inclinacion'], s['tope_m']

## 5. Validación manual
Comparar `tope_m` y el perfil con las imágenes PNG de perfil latitudinal/longitudinal del SMN.